In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Consolidating multiple systems

Demonstration of how to migrate funds from multiple source systems into LUSID

Attributes
----------
instruments
holdings
transaction configuration
properties
quotes
reconciliations
aggregation
"""

toggle_code("Toggle docstring")

## The Challenge

Many investment firms are working with a complex patchwork of legacy systems that have been introduced and added to over many years, as a result of mergers or to support specific business areas and / or asset classes. 

Each system has its own data model, frequency of updates and idiosyncrasies, making it challenging to provide an aggregated view of all your investment portfolios.

## The Solution

LUSID is an open platform and is API driven, letting you connect both source and downstream systems. LUSID provides a flexible way to load your existing data in without significant data mapping, for example transaction types and instrument modelling. This allows existing users of the data to be repointed to LUSID with no risk of being affected by the migration, and take advantage of new capabilities in a gradual and controlled way.

To demonstrate this capability you will migrate a single fund from multiple source systems into LUSID. You have three systems to migrate across, each system specialises in a different asset class. You have a system for bonds, a system for equities and a system for options. To achieve this migration you will:

1) Create your instrument universe using a range of identifiers

2) Set up a segregated scope for each source system

3) Create a portfolio for your fund in each of your scopes

4) Set your initial holdings

5) Set your transaction type configuration

6) Load a day's worth of transactions

7) Attempt to get your updated holdings

8) Update your transaction type configuration & try again to get your updated holdings

9) Load your end of day positions

10) Perform a bi-temporal reconciliation to ensure there are no breaks

11) Group the views from each system together to create an overall view of your fund

12) Get the value of your fund

First things first though import the libraries that you need and authenticate your LUSID client.

*Run the cell below to import the libraries and authenticate your LUSID client*

In [ ]:
# Import Libraries
import pprint

import pytz
import printer as prettyprint
import pandas as pd
import numpy as np
import json
import uuid
import os

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.cocoon.utilities import create_scope_id
from datetime import datetime, timedelta, time

# Import LUSID
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
import lusid_sample_data as import_data


# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

print("LUSID Environment Initialised")
print(
    "LUSID SDK Version: ",
    api_factory.build(lu.ApplicationMetadataApi)
    .get_lusid_versions()
    .build_version,
)

Your set up now looks like the below. As you progress through the migration you will see updated diagrams of the state of the system.

![Init](img/multiplesystems-initialstate.gif)

## 1) Create your Instrument Universe

Before you can take on any holdings or make any trades you need to ensure that your instrument universe has been populated. In this case you will import your instrument universe from a number of CSV files, one for each source system. Read more about instruments in LUSID in the [LUSID Knowledge Base: Instruments](https://support.lusid.com/what-is-an-instrument).

*Run the cells below to import your instrument universe from your source systems*

In [ ]:
equity_instruments = pd.read_csv("data/multiplesystems-instruments-equities.csv")
equity_instruments.head()

In [ ]:
bond_instruments = pd.read_csv("data/multiplesystems-instruments-bonds.csv")
bond_instruments.head()

In [ ]:
option_instruments = pd.read_csv("data/multiplesystems-instruments-options.csv")
option_instruments.head()

Now that you have the details for your instruments you can go ahead and create an instrument definition for each instrument. These can then be upserted into LUSID. Read about instrument definitions here [LUSID Knowledge Base: What is an Instrument?](https://support.lusid.com/what-is-an-instrument).

As part of this definition you will attach both unique and non-unique identifiers to your instruments. Read more about identifiers here [LUSID Knowledge Base: Which Instrument Identifier Scheme Should I Use?](https://support.lusid.com/which-instrument-identifier-schemes-should-i-use-with-lusid).

You use an upsert method to add instrument definitions to the instrument universe in LUSID. Read more about the behaviour of the upsert method here [LUSID Knowledge Base: Upsert](https://support.lusid.com/upsert-command).

For further usage of the upsert instruments API call refer to the [LUSID API Docs: Upserting Instruments](https://docs.lusid.com/#operation/UpsertInstruments).

As each source system uses different identifiers for the instruments you will need to create a function which allows you to easily select the identifiers that you would like to use in the instrument definitions for the instruments from each source system.

*Run the cell below to create a function for upserting instruments from any one of your sources*

In [ ]:
def upsert_instruments(instrument_universe=None, identifier_columns=None):
    """
    This function upserts instruments from a dataframe

    Keyword arguments:
    instrument_universe (Pandas DataFrame) -- The imported instrument universe
    identifer_columns (Tuple (string, string)) -- The identifier name in CSV file
    and identifier name in LUSID

    Returns:
    N/A
    """

    # Initialise your batch upsert request
    batch_upsert_request = {}

    # Using your instrument universe create your batch request
    for index, instrument in instrument_universe.iterrows():

        # Create your identifiers
        identifiers = {}
        for identifier in identifier_columns:
            identifiers[identifier[1]] = models.InstrumentIdValue(
                value=instrument[identifier[0]]
            )

        # Build your request and add it to the dictionary
        batch_upsert_request[
            instrument["InstrumentName"]
        ] = models.InstrumentDefinition(
            name=instrument["InstrumentName"], identifiers=identifiers
        )

    # Call LUSID to upsert your instrument defintions
    instrument_response = api_factory.build(
        lu.InstrumentsApi
    ).upsert_instruments(request_body=batch_upsert_request)

    # Pretty print the response
    prettyprint.instrument_response(instrument_response)

Now that you have a function to upsert your instruments you can upsert the instruments from each source.

For bonds you have access to an ISIN, client internal identifier and a ticker. 

*Run the cell below to upsert your bond instruments into LUSID*

In [ ]:
# Upsert your bond instruments into LUSID
upsert_instruments(
    instrument_universe=bond_instruments,
    identifier_columns=[
        ("ClientInternal", "ClientInternal"),
        ("Isin", "Isin"),
        ("Ticker", "Ticker"),
    ],
)

For equities you have access to an ISIN, FIGI, client internal identifier and a ticker.

*Run the cell below to upsert your equity instruments into LUSID*

In [ ]:
# Upsert your equity instruments into LUSID
upsert_instruments(
    instrument_universe=equity_instruments,
    identifier_columns=[
        ("ClientInternal", "ClientInternal"),
        ("Isin", "Isin"),
        ("Figi", "Figi"),
        ("Ticker", "Ticker"),
    ],
)

For options you have access to a FIGI, client internal identifier and a ticker.

*Run the cell below to upsert your option instruments into LUSID*

In [ ]:
# Upsert your option instruments into LUSID
upsert_instruments(
    instrument_universe=option_instruments,
    identifier_columns=[
        ("ClientInternal", "ClientInternal"),
        ("Figi", "Figi"),
        ("Ticker", "Ticker"),
    ],
)

You have now upserted all of your instruments into the LUSID instrument master.

![Init](img/multiplesystems-instrumentmaster.gif)

## 2) Set up a Scope for each Source System

To segregate the position and transaction data from each source system you can use scopes. Read more about scopes here [LUSID Knowledge Base: Scopes](https://support.lusid.com/what-is-a-scope-in-lusid-and-how-is-it-used).

You will create a scope for each source system.

*Run the cell below to create a scope for each system*

In [ ]:
bonds_scope = "bonds_system_{}".format(create_scope_id(use_uuid=True))
equities_scope = "equities_system_{}".format(create_scope_id(use_uuid=True))
options_scope = "options_system_{}".format(create_scope_id(use_uuid=True))

scopes = [bonds_scope, equities_scope, options_scope]

prettyprint.heading("Bonds Scope", bonds_scope)
prettyprint.heading("Equities Scope", equities_scope)
prettyprint.heading("Options Scope", options_scope)

![Init](img/multiplesystems-scopes.gif)

## 3) Create your Portfolio in Each Scope

To hold the positions and transactions of the fund you need to create a portfolio in each scope.

Note that every portfolio can be referenced by a unique code. Read more about portfolios in the [LUSID Knowledge Base: Portfolios](https://support.lusid.com/knowledgebase/article/KA-01848).

For further usage of the create portfolio API call refer to the [LUSID API Docs: Create Portfolio](https://docs.lusid.com/#operation/CreatePortfolio).

Note that when you create the portfolios in the cell below you are creating it with a 'created' date of 1052 days ago. This number is rather arbitrary, in practice it should be the date the portfolio came into existence regardless of the system you first created it in, read more about the importance of the created date on a portfolio in the [LUSID Knowledge Base: Importance of Portfolio Creation Date](https://support.lusid.com/importance-of-portfolio-creation-date).

*Run the cell below to create your portfolios*

In [ ]:
# Set the code of your portfolio
portfolio_code = str(uuid.uuid4())

# Set the creation date of your portfolio 
portfolio_creation_date = (datetime.now(pytz.UTC) - timedelta(days=1052)).isoformat()

# Build your request to create your portfolio
request = models.CreateTransactionPortfolioRequest(
    display_name="Global Strategies Fund",
    code=portfolio_code,
    base_currency="USD",
    description="The global strategies fund",
    created=portfolio_creation_date,
    corporate_action_source_id=None,
    accounting_method="AverageCost",
    sub_holding_keys=None,
    properties=None,
)

# Iterate over the scopes for the source systems
for scope in scopes:

    # Call LUSID to create your portfolio
    response = api_factory.build(lu.TransactionPortfoliosApi).create_portfolio(
        scope=scope, create_transaction_portfolio_request=request
    )

    # Pretty print the response
    prettyprint.portfolio_response(response)

You now have an empty portfolio inside each scope.

![Init](img/multiplesystems-portfolios.gif)

## 4) Set your Initial Holdings

Now that you have your instrument universe populated and portfolios created you can load your initial holdings into your portfolio. In this case you will also import your holdings from CSV files. 

*Run the cells below to import your holdings*

In [ ]:
bond_holdings = pd.read_csv("data/multiplesystems-holdings-bonds.csv")
bond_holdings.head()

In [ ]:
equity_holdings = pd.read_csv("data/multiplesystems-holdings-equities.csv")
equity_holdings.head()

In [ ]:
option_holdings = pd.read_csv("data/multiplesystems-holdings-options.csv")
option_holdings.head()

As each source system uses different identifiers for the holdings you will need to create a function which allows you to easily select the identifiers that you would like to use in the identifying the instrument related to each holding.

Read more about how making an adjustment or setting the holdings on a portfolio affects it here [LUSID Knowledge Base: The effect of holding adjustments](https://support.lusid.com/knowledgebase/article/KA-01883).

For further usage of the set holdings API call refer to the [LUSID API Docs: Set Holdings](https://docs.lusid.com/#operation/SetHoldings).

*Run the cell below to create a function to handle adding your holdings*

In [ ]:
def load_holdings(
    holdings, scope, code, holdings_effective_date, instrument_identifier
):
    """
    This function takes a set of holdings from a CSV file and sets them on the appropriate
    portfolio

    Keyword arguments:
    holdings (Pandas DataFrame) -- The imported holdings
    scope (string) -- The scope of the portfolio to set the holdings on
    code (string) -- The code of the portfolio to set the holdings on
    holdings_effective_date (datetime) - The timezone aware datetime from which the
    holdings should be effective
    instrument_identifier (string) -- The name of the identifier to use to resolve
    each holding to an instrument in the instrument master

    Returns:
    N/A
    """

    # Iterate the portfolios in the holdings CSV, note in this case you only have one
    for portfolio in holdings["PortfolioCode"].unique():
        # Initialise a list to hold your adjustments
        holding_adjustments = []

        # Iterate over the holdings in each portfolio
        for index, holding in holdings.loc[
            holdings["PortfolioCode"] == portfolio
        ].iterrows():

            # Set your instrument identifiers based on whether or not instrument is cash
            if "Cash" in holding["InstrumentName"]:
                identifier_key = "Instrument/default/Currency"
                identifier = holding["InstrumentName"].split("_")[0]
            else:
                identifier_key = "Instrument/default/{}".format(instrument_identifier)
                identifier = holding[instrument_identifier]

            # Create your holding adjustment and append it to your list
            holding_adjustments.append(
                models.AdjustHoldingRequest(
                    instrument_identifiers={identifier_key: identifier},
                    tax_lots=[
                        models.TargetTaxLotRequest(
                            units=holding["Quantity"],
                            cost=models.CurrencyAndAmount(
                                amount=holding["Quantity"] * holding["Price"],
                                currency=holding["Currency"],
                            ),
                            portfolio_cost=holding["Quantity"] * holding["Price"],
                            price=holding["Price"],
                        )
                    ],
                )
            )

        # Call LUSID to set your initial holdings
        response = api_factory.build(lu.TransactionPortfoliosApi).set_holdings(
            scope=scope,
            code=code,
            effective_at=holdings_effective_date,
            adjust_holding_request=holding_adjustments,
        )

        # Pretty print our response from LUSID
        prettyprint.set_holdings_response(response, scope, portfolio)

Now that you have a function to set your holdings you can set them for each source. 

These holdings will be effective as of 2 days ago. 

For bonds you will use the internal client identifier. 

Run the cell below to set your bond holdings for your portfolio.

*Run the cell below to set your bond holdings in LUSID*

In [ ]:
# Make the holdings effective from two days ago
holdings_effective_date = (datetime.now(pytz.UTC) - timedelta(days=2)).isoformat()

load_holdings(
    holdings=bond_holdings,
    scope=bonds_scope,
    code=portfolio_code,
    holdings_effective_date=holdings_effective_date,
    instrument_identifier="ClientInternal",
)

For equities you will use the FIGI identifier.

Run the cell below to set your equity holdings for your portfolio.

*Run the cell below to set your equity holdings in LUSID*

In [ ]:
load_holdings(
    holdings=equity_holdings,
    scope=equities_scope,
    code=portfolio_code,
    holdings_effective_date=holdings_effective_date,
    instrument_identifier="Figi",
)

In [ ]:
load_holdings(
    holdings=option_holdings,
    scope=options_scope,
    code=portfolio_code,
    holdings_effective_date=holdings_effective_date,
    instrument_identifier="Figi",
)

![Init](img/multiplesystems-holdings.gif)

## 5) Set your Transaction Type Configuration

Now that you have set your initial holdings you are ready to take on some transactions. Before you can add these trades to LUSID you need to set up a transaction type configuration. This allows the LUSID movement engine to correctly decompose each transaction into its underlying economic movements. Read more about configuring transaction types here [LUSID Knowledge Base: Configuring Transaction Types](https://support.lusid.com/configuring-transaction-types) and movements in LUSID here [LUSID Knowledge Base: Movements](https://support.lusid.com/what-is-a-movement-in-lusid). 

You will import the default LUSID transaction type configuration from a JSON file and then use this to set your configuration inside LUSID.

*Run the cell below to import the LUSID default transaction mapping configuration*

In [ ]:
# Import the default transaction type configuration and load it into a dictionary
default_transaction_mapping = open("data/default_transaction_mapping.json").read()
default_transaction_mapping = json.loads(default_transaction_mapping)
# Pretty print your configuration
pprint.pprint(default_transaction_mapping)

Now that you have imported the default configuration you can build your request to set it inside LUSID.

For further usage of the build transactions API call refer to the [LUSID API Docs: Set Configuration Transaction Types](https://docs.lusid.com/#operation/SetConfigurationTransactionTypes).

*Run the cell below to set the transaction type configuration*

In [ ]:
transaction_configuration_api = api_factory.build(lu.TransactionConfigurationApi)

# Add default side definitions to the non-default transaction type scopes.
default_side_definitions = [
    models.SidesDefinitionRequest(
        side="Side1", 
        side_request=models.SideDefinitionRequest(
            security="Txn:LusidInstrumentId",
            currency="Txn:TradeCurrency",
            rate="Txn:TradeToPortfolioRate",
            units="Txn:Units",
            amount="Txn:TradeAmount")),
    models.SidesDefinitionRequest(
        side="Side2", 
        side_request=models.SideDefinitionRequest(
            security="Txn:SettleCcy",
            currency="Txn:SettlementCurrency",
            rate="SettledToPortfolioRate",
            units="Txn:TotalConsideration",
            amount="Txn:TotalConsideration"))
]
for scope in scopes:
    transaction_configuration_api.set_side_definitions(default_side_definitions, scope = scope)

# Helper functions
def map_properties(properties):
    return {property["key"]: models.PerpetualProperty(key=property["key"], value=models.PropertyValue(label_value=property["value"])) for property in properties}
def map_alias(alias):
    return models.TransactionTypeAlias(type=alias["type"], description=alias["description"], transaction_class=alias["transactionClass"], transaction_roles=alias["transactionRoles"])
def map_movement(movement):
    return models.TransactionTypeMovement(movement_types=movement["movementTypes"], side=movement["side"], direction=movement["direction"], properties=map_properties(movement["properties"]))
def map_transaction_type_request(transaction_type_request):
    return models.TransactionTypeRequest(
        aliases=[map_alias(alias) for alias in transaction_type_request["aliases"]],
        movements=[map_movement(movement) for movement in transaction_type_request["movements"]],
        properties=map_properties(transaction_type_request["properties"]))

# Iterate over your configurations in the default mapping
for configuration in default_transaction_mapping:
    transaction_type_requests = [map_transaction_type_request(transaction_type_request) for transaction_type_request in configuration["transactionTypeRequests"]]
    # In all three scopes, call LUSID to set the configuration for our transaction types and update the transaction type scope of the portfolio
    for scope in scopes:
        transaction_configuration_api.set_transaction_type_source(
            source=configuration["source"],
            transaction_type_request=transaction_type_requests,
            scope=scope
        )
        # Call LUSID to update the transaction type scope of your portfolio 
        patch_document = [
            {
                "value": scope,
                "path": "/transactiontypescope",
                "op": "add"
            }
        ]
        patch_response = api_factory.build(lu.TransactionPortfoliosApi).patch_portfolio_details(
            scope=scope,
            code=portfolio_code,
            operation=patch_document)

![Init](img/multiplesystems-transactiontypes.gif)

## 6) Load your Transactions

Now that you have set your transaction type configuration you are ready to take on some transactions. You have made several trades over the last day that you would like to add to LUSID. You will import these from CSV files. 

*Run the cells below to import your transactions*

In [ ]:
bond_transactions = pd.read_csv("data/multiplesystems-transactions-bonds.csv")
bond_transactions.head()

In [ ]:
equity_transactions = pd.read_csv("data/multiplesystems-transactions-equities.csv")
equity_transactions.head()

In [ ]:
option_transactions = pd.read_csv("data/multiplesystems-transactions-options.csv")
option_transactions.head()

As each source system uses different identifiers for the transactions you will need to create a function which allows you to easily select the identifiers that you would like to use in the identifying the instrument related to each transaction.

Read more about transactions here [LUSID Knowledge Base: Transactions](https://support.lusid.com/what-is-a-transaction). 

For further usage of the upsert transactions API call refer to the [LUSID API Docs: Upsert Transactions](https://docs.lusid.com/#operation/UpsertTransactions).

*Run the cell below to create a function to handle upserting your transactions*

In [ ]:
def load_transactions(
    transactions,
    scope,
    code,
    trade_date,
    settlement_date,
    instrument_identifier,
    source_system,
):
    """
    This function takes a set of trades loaded from a CSV and upserts them into LUSID

    Keyword arguments:
    transactions (Pandas DataFrame) -- The imported transactions
    scope (string) -- The scope of the portfolio to upsert the transactions to
    code (string) -- The code of the portfolio to upsert the transactions to
    trade_date (datetime) - The timezone aware datetime at which the trade occured
    settlement_date (datetime) - The timezone aware datetime at which the trade
    will be settled
    instrument_identifier (string) -- The name of the identifier to use to resolve
    each transaction to an instrument in the instrument master
    source_system (string) -- The source system that the transaction came from

    Returns:
    N/A
    """

    # Iterate the portfolios in the holdings CSV, note in this case you only have one
    for portfolio in transactions["PortfolioCode"].unique():
        # Initialise a list to hold your adjustments
        transactions_requests = []

        # Iterate over the holdings in each portfolio
        for index, transaction in transactions.loc[
            transactions["PortfolioCode"] == portfolio
        ].iterrows():

            # Set your instrument identifiers based on whether or not instrument is cash
            if "Cash" in transaction["InstrumentName"]:
                identifier_key = "Instrument/default/Currency"
                identifier = transaction["InstrumentName"].split("_")[0]
            else:
                identifier_key = "Instrument/default/{}".format(instrument_identifier)
                identifier = transaction[instrument_identifier]

            if transaction["Quantity"] == transaction["SettlementAmt"]:
                price = 1
            else:
                price = transaction["Price"]

            # Create your holding adjustment and append it to your list
            transactions_requests.append(
                models.TransactionRequest(
                    transaction_id=str(transaction["TransactionId"]),
                    type=transaction["TransactionType"],
                    instrument_identifiers={identifier_key: identifier},
                    transaction_date=trade_date,
                    settlement_date=settlement_date,
                    units=transaction["Quantity"],
                    transaction_price=models.TransactionPrice(
                        price=price, type="Price"
                    ),
                    total_consideration=models.CurrencyAndAmount(
                        amount=transaction["SettlementAmt"],
                        currency=transaction["SettlementCurrency"],
                    ),
                    source=source_system,
                    transaction_currency=transaction["TransactionCurrency"],
                )
            )

        # Call LUSID to set your initial holdings
        response = api_factory.build(
            lu.TransactionPortfoliosApi
        ).upsert_transactions(
            scope=scope, code=code, transaction_request=transactions_requests
        )

        # Pretty print our response from LUSID
        prettyprint.transactions_response(response, scope, code)

You may have noticed that there are no transaction or settlement dates in any of the CSV files. In this case all these trades happened yesterday, so that is the trade date. For the settlement date you can make the simplifying assumption that all trades will be settling two days from the trade date.

*Run the cell below to create the trade and settlement dates*

In [ ]:
trade_date = (datetime.now(pytz.UTC) - timedelta(days=1)).isoformat()
settlement_date = (datetime.now(pytz.UTC) + timedelta(days=1)).isoformat()
prettyprint.heading("Trade Date", str(trade_date))
prettyprint.heading("Settlement Date", str(settlement_date))

Now that you have a function to upsert your transactions you can upsert them for each source. 

For bonds you will use the internal client identifier. 

Run the cell below to upsert your bond transactions for your portfolio.

*Run the cell below to upsert your bond transactions into LUSID*

In [ ]:
load_transactions(
    bond_transactions,
    bonds_scope,
    portfolio_code,
    trade_date,
    settlement_date,
    "ClientInternal",
    "Bonds",
)

For equities you will use the FIGI identifier. 

Run the cell below to upsert your equity transactions for your portfolio.

*Run the cell below to upsert your equity transactions into LUSID*

In [ ]:
load_transactions(
    equity_transactions,
    equities_scope,
    portfolio_code,
    trade_date,
    settlement_date,
    "Figi",
    "Equities",
)

For options you will also use the FIGI identifier. 

Run the cell below to upsert your option transactions for your portfolio.

*Run the cell below to upsert your option transactions into LUSID*

In [ ]:
load_transactions(
    option_transactions,
    options_scope,
    portfolio_code,
    trade_date,
    settlement_date,
    "Figi",
    "Options",
)

![Init](img/multiplesystems-trades.gif)

## 7) Get your Holdings

You can see how these transactions have affected your holdings by trying to retrieve the holdings of your portfolio. By configuring your transaction types LUSID should have been able to decompose each transaction into its appropriate economic movements. 

For further usage of the get holdings API call refer to the [LUSID API Docs: Get Holdings](https://docs.lusid.com/#operation/GetHoldings).

*Run the cell below to get our holdings from our portfolio*

In [ ]:
# Try and get your holdings, otherwise provide the error message
try:
    response = api_factory.build(lu.TransactionPortfoliosApi).get_holdings(
        scope=options_scope,
        code=portfolio_code,
        property_keys=["Instrument/default/Name"],
    )
    prettyprint.holdings_response(response, scope, portfolio_code)
except ApiException as e:
    print("Error retrieving holdings as {}".format(e))

Here you can see that it looks like you have a transaction type in your transactions file which is not in your configuration.

![Init](img/multiplesystems-failedholdings.gif)

## 8) Update your Transaction Type Configuration

You can find out what might be the problematic transaction type by listing all of your unique transaction types for each source.

*Run the cell below to find your unique transaction types for bonds*

In [ ]:
bond_transaction_types = pd.read_csv("data/multiplesystems-transactions-bonds.csv")[
    "TransactionType"
].unique()
print(bond_transaction_types)

Here you can see that you have three transaction types in your list of bond transactions, you can check of any are in your current configuration by listing your current configuration in LUSID.

For further usage of the list configuration transaction types API call refer to the [LUSID API Docs: List Transaction Type Configuration](https://docs.lusid.com/#operation/ListConfigurationTransactionTypes).

*Run the cell below to check if any of these transaction types are configured*

In [ ]:
# Call LUSID to get your transaction type configuration
response = transaction_configuration_api.list_transaction_types(scope = bonds_scope)
# Pretty print the configuration
prettyprint.transaction_type_response(response, filters=bond_transaction_types)

It is clear from this result that none of these transaction types currently exist in the default configuration. 

So that LUSID can generate the holdings from the portfolio, you need to add the missing transaction types.

To ensure that there is no collision between the different types for each system and their underlying economic meaning, you can group the types of each source by setting the transaction group parameter. 

For your bond transactions we will set the transaction group to be 'Bonds'. Note that this is linked to the source you used when upserting the transactions. LUSID will look for any transaction types in the 'Bonds' group before falling back to the default transaction types.

*Run the cell below to add the missing transaction types for bonds into the configuration*

In [ ]:
transaction_type_requests = [
    models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="ACCR",
                description="Interest accrued on a bond",
                transaction_class="BondInstruments",
                transaction_roles="LongLonger",
            )
        ],
        movements=[
            models.TransactionTypeMovement(
                movement_types="CashCommitment",
                side="Side2",
                direction=1,
                properties=None,
                mappings=None,
            )
        ],
        properties=None,
    ),
    models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="PURCHASE",
                description="A purchase of a bond",
                transaction_class="BondInstruments",
                transaction_roles="LongLonger",
            )
        ],
        movements=[
            models.TransactionTypeMovement(
                movement_types="StockMovement",
                side="Side1",
                direction=1,
                properties=None,
                mappings=None,
            ),
            models.TransactionTypeMovement(
                movement_types="CashCommitment",
                side="Side2",
                direction=-1,
                properties=None,
                mappings=None,
            ),
        ],
        properties=None,
    ),
    models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="SALE",
                description="The sale of a bond",
                transaction_class="BondInstruments",
                transaction_roles="LongShorter",
            )
        ],
        movements=[
            models.TransactionTypeMovement(
                movement_types="StockMovement",
                side="Side1",
                direction=-1,
                properties=None,
                mappings=None,
            ),
            models.TransactionTypeMovement(
                movement_types="CashCommitment",
                side="Side2",
                direction=1,
                properties=None,
                mappings=None,
            ),
        ],
        properties=None,
    ),
]

upsert_txn_config = transaction_configuration_api.set_transaction_type_source(
    source="Bonds",
    transaction_type_request=transaction_type_requests,
    scope=bonds_scope
)

Now you can take a look at equities.

*Run the cell below to see the unique transaction types for the equity transactions*

In [ ]:
equity_transaction_types = pd.read_csv(
    "data/multiplesystems-transactions-equities.csv"
)["TransactionType"].unique()
print(equity_transaction_types)

Now you can check to see if any of these 4 transaction types exist in the current configuration.

*Run the cell below to identify the missing transaction types for equities*

In [ ]:
# Call LUSID to get your transaction type configuration
response = transaction_configuration_api.list_transaction_types(scope = equities_scope)
# Pretty print the configuration
prettyprint.transaction_type_response(response, filters=equity_transaction_types)

So as you can see all four types are missing from the configuration and need to be added.

*Run the cell below to add the missing types to the Equities group*

In [ ]:
transaction_type_requests=[
    models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="DIV",
                description="A dividend from an equity",
                transaction_class="EquityInstruments",
                transaction_roles="LongLonger",
            )
        ],
        movements=[
            models.TransactionTypeMovement(
                movement_types="CashCommitment",
                side="Side2",
                direction=1,
                properties=None,
                mappings=None
            )
        ],
        properties=None
    ),
    models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="BUY",
                description="The purchase of an equity",
                transaction_class="EquityInstrument",
                transaction_roles="LongLonger"
            )
        ],
        movements=[
            models.TransactionTypeMovement(
                movement_types="StockMovement",
                side="Side1",
                direction=1,
                properties=None,
                mappings=None
            ),
            models.TransactionTypeMovement(
                movement_types="CashCommitment",
                side="Side2",
                direction=-1,
                properties=None,
                mappings=None
            )
        ],
        properties=None
    ),
    models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="SHORTSELL",
                description="A short position created from borrowing and selling an equity",
                transaction_class="EquityInstruments",
                transaction_roles="ShortShorter",
            )
        ],
        movements=[
            models.TransactionTypeMovement(
                movement_types="StockMovement",
                side="Side1",
                direction=-1,
                properties=None,
                mappings=None,
            ),
            models.TransactionTypeMovement(
                movement_types="CashCommitment",
                side="Side2",
                direction=1,
                properties=None,
                mappings=None,
            ),
        ],
        properties=None,
    ),
    models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="SELL",
                description="A sale of an equity",
                transaction_class="EquityInstruments",
                transaction_roles="LongShorter",
            )
        ],
        movements=[
            models.TransactionTypeMovement(
                movement_types="StockMovement",
                side="Side1",
                direction=-1,
                properties=None,
                mappings=None,
            ),
            models.TransactionTypeMovement(
                movement_types="CashCommitment",
                side="Side2",
                direction=1,
                properties=None,
                mappings=None,
            ),
        ],
        properties=None,
    )
]

upsert_txn_config = transaction_configuration_api.set_transaction_type_source(
    source="Equities",
    transaction_type_request=transaction_type_requests,
    scope=equities_scope
)

In [ ]:
option_transaction_types = pd.read_csv("data/multiplesystems-transactions-options.csv")[
    "TransactionType"
].unique()
print(option_transaction_types)

Here you can see that there is only a single transaction type.

*Run the cell below to check if the option transaction type exists in the configuration*

In [ ]:
# Call LUSID to get your transaction type configuration
response = transaction_configuration_api.list_transaction_types(scope=options_scope)
# Pretty print the configuration
prettyprint.transaction_type_response(response, filters=option_transaction_types)

It looks like this is also missing from the configuration and needs to be added.

*Run the cell below to add the missing types to the Options group*

In [ ]:
new_transaction_config=models.TransactionTypeRequest(
    aliases=[
        models.TransactionTypeAlias(
            type="CALLCONTRACT",
            description="The purchase of a call options contract",
            transaction_class="EquityInstrument",
            transaction_roles="LongLonger"
        )
    ],
    movements=[
        models.TransactionTypeMovement(
            movement_types="StockMovement",
            side="Side1",
            direction=1,
            properties=None,
            mappings=None
        ),
        models.TransactionTypeMovement(
            movement_types="CashCommitment",
            side="Side2",
            direction=-1,
            properties=None,
            mappings=None
        )
    ],
    properties=None
)



upsert_txn_config = transaction_configuration_api.set_transaction_type(
    source="Options",
    type="CALLCONTRACT",
    transaction_type_request=new_transaction_config,
    scope=options_scope
)

So that you can easily check the last time that you updated your transaction type configuration, you can set a property on your portfolio with the date of the last change. Read more about properties here [LUSID Knowledge Base: Properties](https://support.lusid.com/what-is-a-property). 

For further usage of the create property definition API call refer to the [LUSID API Docs: Create Property Definition](https://docs.lusid.com/#operation/CreatePropertyDefinition).

*Run the cell below to create a property to hold the date of the last change*

In [ ]:
# Create our request to define a new property
request = models.CreatePropertyDefinitionRequest(
    domain="Portfolio",
    scope=scope,
    code="lastconfigurationchange",
    value_required=False,
    display_name="lastconfigurationchange",
    data_type_id=models.ResourceId(scope="system", code="date"),
)

# Call LUSID to create our new property
response = api_factory.build(
    lu.PropertyDefinitionsApi
).create_property_definition(create_property_definition_request=request)

# Grab the key off the response to use when referencing this property in other LUSID calls
portfolio_property_key = response.key

# Pretty print our key
prettyprint.heading("Portfolio Property Key", portfolio_property_key)

Now you can set the value of this property on your portfolio, as you just changed the configuration you can use the current datetime. We will also update the portfolio's display name.

For further usage of the upsert portfolio properties API call refer to the [LUSID API Docs: Upsert Portfolio Properties](https://docs.lusid.com/#operation/UpsertPortfolioProperties).

*Run the cell below to add the date of the last change to your portfolio*

In [ ]:
# Call LUSID to add the date of the last change to the portfolio
response = api_factory.build(lu.PortfoliosApi).upsert_portfolio_properties(
    scope=scope,
    code=portfolio_code,
    request_body={
        portfolio_property_key: models.ModelProperty(
            key=portfolio_property_key,
            value=models.PropertyValue(label_value=datetime.now(pytz.UTC).isoformat()),
        )
    },
)

# Pretty print the response
prettyprint.portfolio_properties_response(response)

# Update the name of the portfolio as well
response = api_factory.build(lu.PortfoliosApi).update_portfolio(
    scope=scope,
    code=portfolio_code,
    effective_at=portfolio_creation_date,
    update_portfolio_request=models.UpdatePortfolioRequest(
        display_name="Global Strategies Fund - Options"
    ),
)

Now that you have configured the transaction types, you can try again to generate the holdings for your portfolio

*Run the cell below to try and generate the holdings for the fund based on the options source*

In [ ]:
# Try and get your holdings, otherwise provide the error message
try:
    response = api_factory.build(lu.TransactionPortfoliosApi).get_holdings(
        scope=options_scope,
        code=portfolio_code,
        property_keys=["Instrument/default/Name"],
    )
    holdings = prettyprint.get_holdings_df(response)
except ApiException as e:
    print("Error retrieving holdings as {}".format(e))

holdings

![Init](img/multiplesystems-updatetransactiontypes.gif)

## 9) Load your End of Day Positions

Now that you are able to load your transactions into LUSID and generate the position of your fund, you need to ensure that the positions are aligned with the source systems and that there are no discrepancies. 

To do this you need to load the end of day positions into LUSID from each source system. You can use the same load holdings function you used earlier to set the initial positions.

*Run the cells below to load the end of day holdings and set them on your portfolio for each system*

In [ ]:
# Make the holdings effective from two days ago
end_of_day_effective_date = datetime.now(pytz.UTC) - timedelta(days=0.5)

holdings = pd.read_csv("data/multiplesystems-holdings-eod-bonds.csv")
holdings.head()

In [ ]:
load_holdings(
    holdings=holdings,
    scope=bonds_scope,
    code=portfolio_code,
    holdings_effective_date=end_of_day_effective_date.isoformat(),
    instrument_identifier="ClientInternal",
)

In [ ]:
holdings = pd.read_csv("data/multiplesystems-holdings-eod-equities.csv")
holdings.head()

In [ ]:
load_holdings(
    holdings=holdings,
    scope=equities_scope,
    code=portfolio_code,
    holdings_effective_date=end_of_day_effective_date.isoformat(),
    instrument_identifier="Figi",
)

In [ ]:
holdings = pd.read_csv("data/multiplesystems-holdings-eod-options.csv")
holdings.head()

In [ ]:
load_holdings(
    holdings=holdings,
    scope=options_scope,
    code=portfolio_code,
    holdings_effective_date=end_of_day_effective_date.isoformat(),
    instrument_identifier="Figi",
)

## 10) Perform a Bi-Temporal Reconciliation 

Now you can make use of LUSID's bi-temporal data store to compare your portfolio before and after adjusting for the end of day position. You can do this by performing a bi-temporal reconciliation. Read more about reconciliations here [LUSID Knowledge Base: Performing a Reconciliation](https://support.lusid.com/knowledgebase/article/KA-01881).

By looking for reconciliation breaks between your portfolio before it was adjusted and after it was adjusted you can see what discrepancies have arisen from any issues with the transaction feed during the day. 

For further usage of the reconcile holdings API call refer to the [LUSID API Docs: Reconcile Holdings](https://docs.lusid.com/#operation/ReconcileValuation).

First you can check that the positions in LUSID are aligned with the bond source system.

*Run the cell below to reconcile your portfolio in the bonds scope with itself before and after the adjustment*

In [ ]:
# Build your left side of the reconciliation before you made the adjustment
before_adjustment = models.PortfolioReconciliationRequest(
    portfolio_id=models.ResourceId(scope=bonds_scope, code=portfolio_code),
    effective_at=(end_of_day_effective_date - timedelta(hours=1)).isoformat(),
    as_at=datetime.now(pytz.UTC).isoformat(),
)

# Build your right side of the reconciliation after you made the adjustment
after_adjustment = models.PortfolioReconciliationRequest(
    portfolio_id=models.ResourceId(scope=bonds_scope, code=portfolio_code),
    effective_at=(end_of_day_effective_date + timedelta(hours=1)).isoformat(),
    as_at=datetime.now(pytz.UTC).isoformat(),
)

# Create your reconciliation request
reconcile_holdings_request = models.PortfoliosReconciliationRequest(
    left=before_adjustment,
    right=after_adjustment,
    instrument_property_keys=["Instrument/default/Name", "Instrument/default/Figi"],
)

# Reconcile your two portfolios
response = api_factory.build(lu.ReconciliationsApi).reconcile_holdings(
    portfolios_reconciliation_request=reconcile_holdings_request
)

# Pretty print the response
prettyprint.reconciliation_response(response, bonds_scope, portfolio_code)

Next you can look at equities.

*Run the cell below to reconcile your portfolio in the equities scope with itself before and after the adjustment*

In [ ]:
# Build your left side of the reconciliation before you made the adjustment
before_adjustment = models.PortfolioReconciliationRequest(
    portfolio_id=models.ResourceId(scope=equities_scope, code=portfolio_code),
    effective_at=(end_of_day_effective_date - timedelta(hours=1)).isoformat(),
    as_at=datetime.now(pytz.UTC).isoformat(),
)

# Build your right side of the reconciliation after you made the adjustment
after_adjustment = models.PortfolioReconciliationRequest(
    portfolio_id=models.ResourceId(scope=equities_scope, code=portfolio_code),
    effective_at=(end_of_day_effective_date + timedelta(hours=1)).isoformat(),
    as_at=datetime.now(pytz.UTC).isoformat(),
)

# Create your reconciliation request
reconcile_holdings_request = models.PortfoliosReconciliationRequest(
    left=before_adjustment,
    right=after_adjustment,
    instrument_property_keys=["Instrument/default/Name", "Instrument/default/Figi"],
)

# Reconcile your two portfolios
response = api_factory.build(lu.ReconciliationsApi).reconcile_holdings(
    portfolios_reconciliation_request=reconcile_holdings_request
)

# Pretty print the response
prettyprint.reconciliation_response(response, equities_scope, portfolio_code)

Finally you can look at options.

*Run the cell below to reconcile your portfolio in the options scope with itself before and after the adjustment*

In [ ]:
# Build your left side of the reconciliation before you made the adjustment
before_adjustment = models.PortfolioReconciliationRequest(
    portfolio_id=models.ResourceId(scope=options_scope, code=portfolio_code),
    effective_at=(end_of_day_effective_date - timedelta(hours=1)).isoformat(),
    as_at=datetime.now(pytz.UTC).isoformat(),
)

# Build your right side of the reconciliation after you made the adjustment
after_adjustment = models.PortfolioReconciliationRequest(
    portfolio_id=models.ResourceId(scope=options_scope, code=portfolio_code),
    effective_at=(end_of_day_effective_date + timedelta(hours=1)).isoformat(),
    as_at=datetime.now(pytz.UTC).isoformat(),
)

# Create your reconciliation request
reconcile_holdings_request = models.PortfoliosReconciliationRequest(
    left=before_adjustment,
    right=after_adjustment,
    instrument_property_keys=["Instrument/default/Name", "Instrument/default/Figi"],
)

# Reconcile your two portfolios
response = api_factory.build(lu.ReconciliationsApi).reconcile_holdings(
    portfolios_reconciliation_request=reconcile_holdings_request
)

# Pretty print the response
prettyprint.reconciliation_response(response, options_scope, portfolio_code)

![Init](img/multiplesystems-reconciliation.gif)

## 11) Group the Sources to Create an Overall View of your Fund

Now that you've ensured that the holdings in LUSID match those in the source systems, you can be confident that LUSID has correctly ingested and processed the day's transactions. 

So far your fund has been split up by the different source systems. To get an overall view of your fund you can now combine the portfolio in the bonds scope, equities scope and options scope. 

To do this you can make use of portfolio groups.

Read more about portfolio groups here [LUSID Knowledge Base: How do you Group and Aggregate Portfolios?](https://support.lusid.com/how-do-you-group-and-aggregate-portfolios)

*Run the cell below to create a new scope and a portfolio group for the combined fund*

In [ ]:
fund_scope = "combined_systems_{}".format(str(uuid.uuid4())[:4])

group_request = models.CreatePortfolioGroupRequest(
    code=portfolio_code + "-Group",
    display_name=portfolio_code + "Group",
    created=portfolio_creation_date,
)

response = api_factory.build(lu.PortfolioGroupsApi).create_portfolio_group(
    scope=fund_scope, create_portfolio_group_request=group_request
)

prettyprint.portfolio_group_response(response, "created")

You can now add the portfolio from each scope to the group.

*Run the cell below to add each of the portfolios representing the different source systems into a single group*

In [ ]:
for scope in scopes:

    response = api_factory.build(lu.PortfolioGroupsApi).add_portfolio_to_group(
        scope=fund_scope,
        code=portfolio_code + "-Group",
        resource_id=models.ResourceId(scope=scope, code=portfolio_code),
        effective_at=response.created.isoformat(),
    )

    prettyprint.get_portfolio_group_response(response)

![Portfolio Group](img/multiplesystems-portfoliogroup.gif)

## 12) Value your Fund

To value a portfolio in LUSID you need to upsert market data quotes against the underlying holdings or specify an analytics library to use. Read more about aggregating and valuing portfolios in the [LUSID Knowledge Base: Aggregations and Valuations](https://support.lusid.com/knowledgebase/category/?id=CAT-01051).

In this case you will upsert market data quotes to the quote store to be used in an aggregation request. You will import these quotes from a CSV file.

*Run the cell below to import the market data prices*

In [ ]:
prices = pd.read_csv("data/multiplesystems-prices.csv")
prices.head(n=50)

Now that you have imported the market data you can add it to the quote store in LUSID. Read more about what a quote is in the [LUSID Knowledge Base: What is a Quote?](https://support.lusid.com/what-is-a-quote).

For further usage of the Upsert Quotes API call refer to the [LUSID API Docs: Upsert Quotes](https://docs.lusid.com/#operation/UpsertQuotes).

Note that the quotes locations must match the rules used when valuation is run in order for them to be found. The two should be thought of as a pair.
*Run the cell below to upsert the market data quotes into LUSID*

In [ ]:
instrument_quotes = {}

for index, quote in prices.iterrows():

    if type(quote["figi"]) is str:
        instrument_id = quote["figi"]
        instrument_id_type = "Figi"
    elif type(quote["client_internal"]) is str:
        instrument_id = quote["client_internal"]
        instrument_id_type = "ClientInternal"
    else:
        instrument_id = quote["isin"]
        instrument_id_type = "Isin"

    luid = (
        api_factory.build(lu.SearchApi)
        .instruments_search(
            instrument_search_property=[
                models.InstrumentSearchProperty(
                    key="Instrument/default/{}".format(instrument_id_type),
                    value=instrument_id,
                )
            ],
            mastered_only=True,
        )[0]
        .mastered_instruments[0]
        .identifiers["LusidInstrumentId"]
        .value
    )

    instrument_quotes[luid] = models.UpsertQuoteRequest(
        quote_id=models.QuoteId(
            quote_series_id=models.QuoteSeriesId(
                provider="DataScope",
                price_source="",
                instrument_id=luid,
                instrument_id_type="LusidInstrumentId",
                quote_type="Price",
                field="Mid",
            ),
            effective_at=end_of_day_effective_date.isoformat(),
        ),
        metric_value=models.MetricValue(
            value=quote["price_current"], unit=quote["currency"]
        ),
        lineage="InternalSystem",
    )

response = api_factory.build(lu.QuotesApi).upsert_quotes(
    scope=fund_scope, request_body=instrument_quotes
)

prettyprint.upsert_quotes_response(response)

Some of the portfolios hold instruments that have a local currency of GBP which differs from the portfolios base currency of USD. Because of this we also need the appropriate FX rates in the quotes store.

*Run the cell below to add a quote for the GBP/USD exchange rate*

In [ ]:
instrument_quote = models.UpsertQuoteRequest(
    quote_id=models.QuoteId(
        quote_series_id=models.QuoteSeriesId(
            provider="DataScope",
            instrument_id="GBP/USD",
            instrument_id_type="CurrencyPair",
            quote_type="Rate",
            field="Mid",
        ),
        effective_at=end_of_day_effective_date.isoformat(),
    ),
    metric_value=models.MetricValue(value=1.26, unit="GBP/USD"),
    lineage="InternalSystem",
)

response = api_factory.build(lu.QuotesApi).upsert_quotes(
    scope=fund_scope, request_body={"GBP/USD": instrument_quote}
)

prettyprint.upsert_quotes_response(response)

![Quotes Store](img/multiplesystems-quotesstore.gif)

Now that the quotes have been added to the quote store you can aggregate your base fund portfolio. The logic for an aggregation is controlled by a LUSID recipe. Read more about recipes in the [LUSID Knowledge Base: What is a Recipe and How Are They Used?](https://support.lusid.com/knowledgebase/article/KA-01895).

For further usage of the Get Aggregation by Portfolio API call refer to the [LUSID API Docs: Get Aggregation by Portfolio](https://docs.lusid.com/#operation/GetAggregationByPortfolio).

*Run the cell below to aggregate and value the base fund*

In [ ]:
recipe_scope = "multi-system"
recipe_code = "market_value"

inline_recipe = models.ConfigurationRecipe(
    scope=recipe_scope,
    code=recipe_code,
    market=models.MarketContext(
        market_rules=[
            models.MarketDataKeyRule(
                key="Quote.Figi.*",
                supplier="DataScope",
                data_scope=fund_scope,
                quote_type="Price",
                field="Mid",
            ),
            models.MarketDataKeyRule(
                key="Quote.Isin.*",
                supplier="DataScope",
                data_scope=fund_scope,
                quote_type="Price",
                field="Mid",
            ),
            models.MarketDataKeyRule(
                key="Quote.LusidInstrumentId.*",
                supplier="DataScope",
                data_scope=fund_scope,
                quote_type="Price",
                field="Mid",
            ),
            models.MarketDataKeyRule(
                key="Fx.CurrencyPair.*",
                supplier="DataScope",
                data_scope=fund_scope,
                quote_type="Rate",
                field="Mid",
            ),
        ],
        suppliers=models.MarketContextSuppliers(
            commodity="DataScope",
            credit="DataScope",
            equity="DataScope",
            fx="DataScope",
            rates="DataScope",
        ),
        options=models.MarketOptions(
            default_supplier="DataScope",
            default_instrument_code_type="Figi",
            default_scope=fund_scope,
        ),
    ),
)

# Upsert recipe to LUSID
upsert_recipe_request = models.UpsertRecipeRequest(configuration_recipe=inline_recipe)
response = api_factory.build(
    lu.ConfigurationRecipeApi
).upsert_configuration_recipe(upsert_recipe_request)

valuation_request = models.ValuationRequest(
    recipe_id=models.ResourceId(scope=inline_recipe.scope, code=inline_recipe.code),
    metrics=[
        models.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
        models.AggregateSpec(key="Instrument/default/Name", op="Value"),
        models.AggregateSpec(key="Holding/default/Units", op="Sum"),
        models.AggregateSpec(key="Holding/default/Cost", op="Sum"),
        models.AggregateSpec(key="Valuation/PvInReportCcy", op="Sum"),
    ],
    valuation_schedule=models.ValuationSchedule(effective_at=end_of_day_effective_date.isoformat()),
    portfolio_entity_ids=[
        models.PortfolioEntityId(
            scope=fund_scope,
            code=portfolio_code + "-Group",
            portfolio_entity_type="GroupPortfolio",
        )
    ],
    group_by=["Instrument/default/LusidInstrumentId"],
)

response = api_factory.build(lu.AggregationApi).get_valuation(
    valuation_request=valuation_request
)

prettyprint.aggregation_responses_generic_df([response])

![Aggregation Result](img/multiplesystems-aggregationresult.gif)